# Data Extraction for Phase 1
Extract 100 SpamAssassin ham emails and 50 Phishbowl phishing emails

In [1]:
import pandas as pd
import os
import random
from pathlib import Path
import re

## 1. Extract 100 SpamAssassin Ham Emails

In [2]:
def parse_rfc822_email(filepath: str) -> dict:
    """Parse RFC822 email file and extract subject, sender, body."""
    try:
        with open(filepath, 'r', encoding='utf-8', errors='ignore') as f:
            content = f.read()
        
        # Split headers and body at first blank line
        if '\n\n' in content:
            header_section, body_section = content.split('\n\n', 1)
        else:
            header_section = content
            body_section = ""
        
        # Extract subject
        subject_match = re.search(r'^Subject: (.+)$', header_section, re.MULTILINE | re.IGNORECASE)
        subject = subject_match.group(1).strip() if subject_match else "[No Subject]"
        
        # Extract sender (From:)
        from_match = re.search(r'^From: (.+)$', header_section, re.MULTILINE | re.IGNORECASE)
        sender = from_match.group(1).strip() if from_match else "[Unknown]"
        
        # Clean body - remove any remaining header-like lines
        body_lines = body_section.split('\n')
        cleaned_lines = []
        for line in body_lines:
            # Skip lines that look like headers (key: value)
            if not re.match(r'^[A-Za-z-]+:\s+.+', line):
                cleaned_lines.append(line)
            elif len(cleaned_lines) > 0:  # Only skip header-like lines at the beginning
                cleaned_lines.append(line)
        
        body = '\n'.join(cleaned_lines).strip()
        
        return {
            'subject': subject,
            'sender': sender,
            'body': body
        }
    except Exception as e:
        print(f"Error parsing {filepath}: {e}")
        return None

In [3]:
# Get all SpamAssassin ham files
spam_dir = Path('data/raw/spamassassin/easy_ham/easy_ham')
all_files = list(spam_dir.glob('*'))

print(f"Total ham files available: {len(all_files)}")

# Sample 100 random files
random.seed(42)  # For reproducibility
sampled_files = random.sample(all_files, 100)

print(f"Sampling 100 files...")

# Parse emails
ham_emails = []
for i, filepath in enumerate(sampled_files, 1):
    email_data = parse_rfc822_email(str(filepath))
    if email_data:
        email_data['id'] = i
        email_data['filename'] = filepath.name
        ham_emails.append(email_data)
    
    if i % 20 == 0:
        print(f"  Processed {i}/100...")

print(f"\n✅ Successfully extracted {len(ham_emails)} ham emails")

Total ham files available: 2551
Sampling 100 files...
  Processed 20/100...
  Processed 40/100...
  Processed 60/100...
  Processed 80/100...
  Processed 100/100...

✅ Successfully extracted 100 ham emails


In [4]:
# Create DataFrame and validate
ham_df = pd.DataFrame(ham_emails)

print("\n📊 Ham Dataset Summary:")
print(f"Rows: {len(ham_df)}")
print(f"Columns: {', '.join(ham_df.columns)}")
print(f"\n📏 Body Length Stats:")
print(f"  Mean: {ham_df['body'].str.len().mean():.1f} chars")
print(f"  Min: {ham_df['body'].str.len().min()} chars")
print(f"  Max: {ham_df['body'].str.len().max()} chars")

# Check for empty bodies
empty_bodies = (ham_df['body'].str.len() < 10).sum()
if empty_bodies > 0:
    print(f"\n⚠️  {empty_bodies} emails with very short bodies (< 10 chars)")
else:
    print(f"\n✅ All emails have substantial body content")

# Sample
print(f"\n📧 Sample Email:")
print(f"Subject: {ham_df.iloc[0]['subject']}")
print(f"Sender: {ham_df.iloc[0]['sender']}")
print(f"Body (first 200 chars): {ham_df.iloc[0]['body'][:200]}...")


📊 Ham Dataset Summary:
Rows: 100
Columns: subject, sender, body, id, filename

📏 Body Length Stats:
  Mean: 1753.2 chars
  Min: 60 chars
  Max: 16464 chars

✅ All emails have substantial body content

📧 Sample Email:
Subject: Re: New gkrellm 2.0.0, gtk2 version
Sender: Brian Fahrlander <kilroy@kamakiriad.com>
Body (first 200 chars): On Mon, 26 Aug 2002 19:14:54 +0200, Matthias Saou <matthias@egwn.net> wrote:

> Hi all,
> 
> I've repackaged the new gkrellm 2.0.0 which is now ported to gtk2
> (woohoo!). Unfortunately, the plugins a...


In [5]:
# Save to CSV
output_path = 'data/raw/spamassassin_ham_100.csv'
ham_df[['id', 'subject', 'sender', 'body']].to_csv(output_path, index=False)
print(f"✅ Saved to: {output_path}")

✅ Saved to: data/raw/spamassassin_ham_100.csv


## 2. Sample 50 Phishbowl Phishing Emails

In [6]:
# Load full Phishbowl dataset
phishbowl_full = pd.read_csv('data/raw/phishbowl.csv')
print(f"Total Phishbowl emails: {len(phishbowl_full)}")

# Sample 50 random emails
random.seed(42)
phishbowl_sample = phishbowl_full.sample(n=50, random_state=42).reset_index(drop=True)

print(f"Sampled 50 emails")

Total Phishbowl emails: 240
Sampled 50 emails


In [11]:
# Clean HTML from email_message column
import re

def strip_html_tags(html_text: str) -> str:
    """Remove HTML tags from text."""
    if pd.isna(html_text):
        return "[No body content]"
    # Remove HTML tags
    text = re.sub(r'<[^>]+>', '', str(html_text))
    # Remove extra whitespace
    text = re.sub(r'\s+', ' ', text)
    # Decode common HTML entities
    text = text.replace('&lt;', '<').replace('&gt;', '>').replace('&amp;', '&')
    text = text.replace('&quot;', '"').replace('&nbsp;', ' ')
    stripped = text.strip()
    # Return placeholder if empty after stripping
    return stripped if stripped else "[No body content]"

# Apply cleaning
phishbowl_sample['body'] = phishbowl_sample['email_message'].apply(strip_html_tags)
phishbowl_sample['subject'] = phishbowl_sample['title']
phishbowl_sample['sender'] = '[Unknown]'  # Phishbowl doesn't include sender

# Create clean dataset
phishbowl_clean = phishbowl_sample[['subject', 'sender', 'body']].copy()
phishbowl_clean.insert(0, 'id', range(1, 51))

print("✅ Cleaned HTML tags from email bodies")

✅ Cleaned HTML tags from email bodies


In [12]:
# Validate
print("\n📊 Phishbowl Sample Summary:")
print(f"Rows: {len(phishbowl_clean)}")
print(f"Columns: {', '.join(phishbowl_clean.columns)}")
print(f"\n📏 Body Length Stats:")
print(f"  Mean: {phishbowl_clean['body'].str.len().mean():.1f} chars")
print(f"  Min: {phishbowl_clean['body'].str.len().min()} chars")
print(f"  Max: {phishbowl_clean['body'].str.len().max()} chars")

# Sample
print(f"\n📧 Sample Email:")
print(f"Subject: {phishbowl_clean.iloc[0]['subject']}")
print(f"Body (first 200 chars): {phishbowl_clean.iloc[0]['body'][:200]}...")


📊 Phishbowl Sample Summary:
Rows: 50
Columns: id, subject, sender, body

📏 Body Length Stats:
  Mean: 967.6 chars
  Min: 17 chars
  Max: 4598 chars

📧 Sample Email:
Subject: **Emergency: Help Needed for COVID-19 Variant Contact Tracing**
Body (first 200 chars): This phish typically originates from a compromised Cornell account that has since been secured. The sender may appear as "Cornell Alerts <[user]@cornell.edu>". The link within directs to a fake CUWebL...


In [13]:
# Save to CSV
output_path = 'data/raw/phishbowl_50.csv'
phishbowl_clean.to_csv(output_path, index=False)
print(f"✅ Saved to: {output_path}")

✅ Saved to: data/raw/phishbowl_50.csv


## Summary

In [10]:
print("\n" + "="*80)
print("PHASE 1 DATA EXTRACTION COMPLETE")
print("="*80)
print("\n✅ Datasets ready:")
print("  1. spamassassin_ham_100.csv (100 benign emails)")
print("  2. phishbowl_50.csv (50 phishing emails)")
print("  3. plain_llm_phishing.csv (50 phishing emails)")
print("  4. hybrid_vtriad_phishing.csv (50 phishing emails)")
print("\n📊 Total: 250 emails (100 benign + 150 phishing)")
print("\n✅ All datasets use schema: id, subject, sender, body")
print("="*80)


PHASE 1 DATA EXTRACTION COMPLETE

✅ Datasets ready:
  1. spamassassin_ham_100.csv (100 benign emails)
  2. phishbowl_50.csv (50 phishing emails)
  3. plain_llm_phishing.csv (50 phishing emails)
  4. hybrid_vtriad_phishing.csv (50 phishing emails)

📊 Total: 250 emails (100 benign + 150 phishing)

✅ All datasets use schema: id, subject, sender, body
